### Testing GraphBuilder

#### Node classification - One Graph - Cora

In [8]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
import torch

dset = datasets.Planetoid(root=f'./../res/datasets/cora', name='cora')

data = dset[0]
print(data.__class__)

dloader = DataLoader(dataset=data, batch_size=1)
print(dloader.__class__)

train_mask, val_mask, test_mask = TrainValTestSplitter.split('cora', dset)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=dset.num_classes, 
            num_features=dset.num_features, 
            hidden_size=16)

model.set_classification_task(task='node')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)
model.train()
epoch_count = 200
for epoch in range(epoch_count):
    # Clear the gradients
    optimizer.zero_grad()
    # Forward Pass
    out = model(data.x, data.edge_index)
    # Find the Loss
    train_loss = criterion(out[train_mask], data.y[train_mask])    
    # Calculate gradients
    train_loss.backward()
    # Update Weights
    optimizer.step()
    print(f"""Epoch: [{epoch}/{epoch_count}]
    Training loss: {train_loss.item():.4f}""")

    ### VALIDATION
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        val_loss = criterion(out[val_mask], data.y[val_mask])
        _, pred = torch.max(out.data, 1)
        correct = (pred[val_mask] == data.y[val_mask]).float().sum().item()
        acc = correct / val_mask.shape[0]
    
    if (epoch + 1) % 10 == 0:
        print(f"""Epoch: [{epoch+1}/{epoch_count}]
          Training loss: {train_loss.item():.4f},\t Validation loss: {val_loss.item():.4f}
          Accuracy: {acc}""")

<class 'torch_geometric.data.data.Data'>
<class 'torch_geometric.loader.dataloader.DataLoader'>
Epoch: [0/200]
    Training loss: 1.9276
Epoch: [1/200]
    Training loss: 1.9127
Epoch: [2/200]
    Training loss: 1.8949
Epoch: [3/200]
    Training loss: 1.8731
Epoch: [4/200]
    Training loss: 1.8474
Epoch: [5/200]
    Training loss: 1.8175
Epoch: [6/200]
    Training loss: 1.7835
Epoch: [7/200]
    Training loss: 1.7462
Epoch: [8/200]
    Training loss: 1.7064
Epoch: [9/200]
    Training loss: 1.6643
Epoch: [10/200]
          Training loss: 1.6643,	 Validation loss: 1.6413
          Accuracy: 0.3948339483394834
Epoch: [10/200]
    Training loss: 1.6204
Epoch: [11/200]
    Training loss: 1.5751
Epoch: [12/200]
    Training loss: 1.5288
Epoch: [13/200]
    Training loss: 1.4812
Epoch: [14/200]
    Training loss: 1.4322
Epoch: [15/200]
    Training loss: 1.3816
Epoch: [16/200]
    Training loss: 1.3295
Epoch: [17/200]
    Training loss: 1.2759
Epoch: [18/200]
    Training loss: 1.2212
Epo

#### Node Classification - Multiple Graphs - LRGB PascalVOC-SP

In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.GCN import GCN
from rewiring.transform.ReduceDiameter import ReduceDiameter

# Define the GCN model
class GCNModel(nn.Module):
    def __init__(self, num_features, num_classes, hidden_size = 16):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_size)
        self.conv2 = GCNConv(hidden_size, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return torch.log_softmax(x, dim=1)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64

# Load the dataset
print("Loading and transforming training dataset...")
train_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', transform=ReduceDiameter(max_diam=200))
print(len(train_dataset))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print("Load and transformation done!")

val_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='val')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='test')
test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = train_dataset.num_features
num_classes = train_dataset.num_classes
hidden_size = 64
model = GCNModel(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
# model.set_classification_task('graph')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3
model.to(device=device)
for epoch in range(num_epochs):

    epoch_train_loss = []
    epoch_val_loss = []

    # Training loop
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()

        out = model(data.x, data.edge_index)
        loss = criterion(out, data.y)
        loss.backward()
        epoch_train_loss.append(loss.item())

        optimizer.step()

    # Validation loop
    model.eval()
    correct = 0
    sum = len(val_loader.dataset.y)
    for data in val_loader:
        data = data.to(device)
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            val_loss = criterion(out, data.y)
            epoch_val_loss.append(val_loss.item())
            _, pred = torch.max(out.data, 1)
            correct += (pred == data.y).float().sum().item()

    train_loss = np.array(epoch_train_loss).mean().item()
    val_loss = np.array(epoch_val_loss).mean().item()
    acc = correct / sum

    print(f"""Epoch: [{epoch+1}/{num_epochs}] \
        Training loss: {train_loss:.4f} \
        Validation loss: {val_loss:.4f} \
        Accuracy: {acc:.4f}""")

print("Training finished!")
print("Starting testing...")
model.eval()
correct = 0
epoch_test_loss = []
sum = len(test_loader.dataset.y)
print(f'Test element count: {sum}')
for data in test_loader:
    data = data.to(device)
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        test_loss = criterion(out, data.y)
        epoch_test_loss.append(test_loss.item())
        _, pred = torch.max(out.data, 1)
        correct += (pred == data.y).float().sum().item()

acc = correct / sum
test_loss = np.array(epoch_test_loss).mean().item()

print(f"""Loss: {test_loss:.4f} \
        Accuracy: {acc:.4f}""")

Loading and transforming training dataset...
8498
Load and transformation done!


KeyboardInterrupt: 

#### Graph Classification - Proteins

In [13]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
import torch
import numpy as np

data = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS')

val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(data, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=data.num_classes, 
            num_features=data.num_features, 
            hidden_size=16)
model.set_classification_task(task='graph')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)

acc_list = []
train_losses = []
val_losses = []

epoch_count = 200
for epoch in range(1, epoch_count + 1):

    epoch_train_loss = []
    epoch_val_loss = []

    ### TRAINING
    model.train()
    for batch in train_dloader:
        # Clear the gradients
        optimizer.zero_grad()
        # Forward Pass
        out = model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
        # Find the Loss
        train_loss = criterion(out, batch.y)
        epoch_train_loss.append(train_loss.item())    
        # Calculate gradients
        train_loss.backward()
        # Update Weights
        optimizer.step()

    ### VALIDATION
    model.eval()
    correct = 0
    for batch in val_dloader:
        with torch.no_grad():
            out = model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
            val_loss = criterion(out, batch.y)
            epoch_val_loss.append(val_loss.item())
            _, pred = torch.max(out.data, 1)
            correct += (pred == batch.y).float().sum().item()

    acc = correct / val_count
    acc_list.append(acc)

    epoch_train_loss = np.array(epoch_train_loss).mean().item()
    epoch_val_loss = np.array(epoch_val_loss).mean().item()
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
        
    if epoch % 10 == 0:
            print(f"""Epoch: [{epoch}/{epoch_count}] \
        Training loss: {epoch_train_loss:.4f} \
        Validation loss: {epoch_val_loss:.4f} \
        Accuracy: {acc:.4f}""")

Epoch: [10/200]         Training loss: 0.6294         Validation loss: 0.6322         Accuracy: 0.6413
Epoch: [20/200]         Training loss: 0.5573         Validation loss: 0.5844         Accuracy: 0.7040
Epoch: [30/200]         Training loss: 0.5391         Validation loss: 0.5834         Accuracy: 0.7085


KeyboardInterrupt: 